# 00 - Dataset and Partitioning

This notebook explains the reproducible data pipeline for `hospital-ids-hfl`.

The dataset is CIC-IDS2017 network-flow data. We treat labels as binary intrusion-detection labels:

- `0`: benign network flow
- `1`: malicious / attack network flow

The notebooks and scripts simulate hospital ownership after partitioning. The initial Kaggle download and central cleaning step are only used to create the simulated local hospital datasets.

In [ ]:
from pathlib import Path
import os

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(root)
root

## 1. Download

Kaggle credentials should be provided outside the repository, for example with `KAGGLE_API_TOKEN` or `~/.kaggle/kaggle.json`. No credential file is stored in this project.

The command below downloads the public Kaggle mirror into `data/raw/`, which is ignored by Git.

In [ ]:
# Run only if data/raw does not already contain the CIC-IDS2017 CSVs.
# !python scripts/download_kaggle.py

In [ ]:
raw_csvs = sorted(Path('data/raw').glob('*.csv'))
len(raw_csvs), [p.name for p in raw_csvs[:3]]

## 2. Clean

`scripts/prepare_cicids.py` performs the shared cleaning step:

- strips whitespace from column names
- maps labels to binary values
- drops metadata/leakage-prone fields such as IPs, flow IDs, and timestamps
- converts remaining features to numeric values
- replaces infinities with missing values
- writes `data/processed/cicids_clean.parquet`

Train-only imputation and standardization are done in the partitioning script.

In [ ]:
# !python scripts/prepare_cicids.py --raw-dir data/raw --output data/processed/cicids_clean.parquet

In [ ]:
import json
import pandas as pd

processed_meta = Path('data/processed/cicids_clean.metadata.json')
if processed_meta.exists():
    meta = json.loads(processed_meta.read_text())
    print('rows:', meta['rows'])
    print('features:', meta['num_features'])
    pd.Series(meta['attack_distribution']).sort_values(ascending=False)
else:
    print('Processed metadata not found yet.')

## 3. Partition

The six hospital partitions are generated with a deterministic seed. The default seed is `123`.

The split is intentionally non-IID. Each hospital is biased toward different attack families while preserving enough benign/attack rows where possible. The final parquet files contain only numeric features and the binary `label` column.

In [ ]:
# !python scripts/make_partitions.py --seed 123

In [ ]:
rows = []
for meta_path in sorted(Path('data/partitions').glob('*/metadata.json')):
    item = json.loads(meta_path.read_text())
    rows.append({
        'hospital_id': item['hospital_id'],
        'region': item['region'],
        'num_train': item['num_train'],
        'num_val': item['num_val'],
        'num_test': item['num_test'],
        'num_features': item['num_features'],
        'train_benign': item['train_label_counts'].get('0', 0),
        'train_attack': item['train_label_counts'].get('1', 0),
    })
pd.DataFrame(rows)

## 4. Privacy Boundary Check

After partitioning, hospitals should read only their own `data/partitions/<hospital_id>` mount. `shared/` may contain scaler metadata, metrics, and checkpoints, but not raw CSV or parquet hospital rows.

In [ ]:
shared_data_files = list(Path('shared').rglob('*.csv')) + list(Path('shared').rglob('*.parquet'))
shared_data_files